# Amazon Bedrock AgentCore 런타임과 AgentCore 메모리 에이전트

## 개요

이 튜토리얼은 AgentCore 런타임과 AgentCore 메모리를 사용하여 첫 번째 메모리 지원 에이전트를 만드는 방법을 보여줍니다. 세션 내에서 이전 상호작용을 기억하는 간단한 "Hello World" 대화형 에이전트를 구축합니다.

### 튜토리얼 세부 정보


| 항목              | 세부 내용                                                         |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | Hello World / 소개                                                |
| 에이전트 유형       | 단일 대화형 에이전트                                               |
| 에이전트 프레임워크  | Strands Agents                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                       |
| 주요 기능           | AgentCore 런타임, 메모리 통합                                     |
| 예제 난이도         | 초급                                                              |
| 사용 SDK           | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit       |

### 학습 내용

이 튜토리얼에서 학습할 내용:
1. 에이전트를 위한 메모리 리소스 생성 방법
2. AgentCoreMemorySessionManager를 사용한 자동 대화 영속화 방법
3. 에이전트를 AgentCore 런타임에 배포하는 방법
4. 세션 관리로 에이전트를 테스트하는 방법


### 아키텍처

이 Hello World 예제는 메모리 통합이 적용된 AgentCore 런타임에 배포되는 간단한 대화형 에이전트를 보여줍니다:

<div style="text-align:left">
    <img src="RuntimeMemoryIntegration.png" width="90%"/>
</div>

## 0. 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
* Python 3.10+
* AWS 자격 증명 설정
* Amazon Bedrock 모델 접근 권한 (Claude Haiku 4.5)
* Amazon Bedrock AgentCore SDK

먼저 필요한 라이브러리를 설치합니다:

In [ ]:
!pip3 install -qr requirements.txt

### 환경 설정

필요한 라이브러리를 임포트하고 환경을 설정합니다:

In [ ]:
# 임포트
import os
import boto3
import uuid
import logging
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

# 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2') # 에이전트의 AWS 리전
memory_client = MemoryClient(region_name=REGION)

## 1. 메모리 리소스 생성

이 섹션에서는 에이전트가 대화 이력을 저장하기 위한 메모리 리소스를 생성합니다. 메모리는 에이전트가 과거 상호작용을 회상하고, 컨텍스트를 유지하며, 시간이 지남에 따라 더 일관된 응답을 제공할 수 있게 합니다.

이 예제에서는 추가적인 장기 전략 없이 간단한 단기 메모리 리소스를 생성합니다. 메모리는 모든 대화 메시지를 저장하여, AgentCore 런타임에서 세션이 종료된 후에도 세션을 계속할 때 에이전트가 이전 상호작용을 기억할 수 있도록 합니다.

In [ ]:
from botocore.exceptions import ClientError

# 이 리소스의 고유 식별자 생성
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeMemoryAgent_{unique_id}"

try:
    # 전략 없이 메모리 리소스 생성 (단기 메모리만)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리를 위한 전략 없음
        description="Short-term memory for AgentCore Runtime agent",
        event_expiry_days=7, # 단기 메모리 보존 기간
    )
    memory_id = memory['id']
    logger.info(f"메모리 생성 완료: {memory_id}")
except ClientError as e:
    logger.info(f"오류: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하면 해당 ID를 조회
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 표시
    logger.error(f"오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if 'memory_id' in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")

## 2. 메모리 지원 에이전트 생성

이 섹션에서는 메모리 통합을 위한 내장 AgentCoreMemorySessionManager를 사용하여 Strands Agents 프레임워크로 메모리 지원 에이전트를 구축합니다.

> **메모리가 중요한 이유**: AgentCore 런타임의 세션은 일정 시간이 지나면 만료되어 대화 컨텍스트가 삭제됩니다. 대화를 메모리에 저장함으로써, 오랜 휴식 후에도 이전 정보가 세션 간에 유지되어 사용자에게 원활한 경험을 제공합니다.

### 에이전트 기능

에이전트는 다음을 수행합니다:
1. 각 사용자 및 어시스턴트 메시지를 자동으로 메모리에 저장
2. 기존 세션을 계속할 때 과거 대화 이력 조회
3. 동일한 사용자와의 여러 상호작용에 걸쳐 컨텍스트 유지

### 구현의 핵심 구성 요소

#### 1. AgentCoreMemorySessionManager (권장)
다음을 자동으로 처리하는 내장 Strands 통합:
- 각 사용자 및 어시스턴트 메시지를 메모리에 저장
- 기존 세션을 계속할 때 과거 대화 이력 조회
- 세션 및 액터 컨텍스트 관리

#### 2. 에이전트 초기화
`initialize_agent` 함수:
- memory_id, session_id, actor_id로 `AgentCoreMemoryConfig`를 사용하여 메모리 구성
- `AgentCoreMemorySessionManager` 인스턴스 생성
- session_manager로 에이전트 설정

#### 3. 진입점 핸들러
runtime_memory_agent 함수:
- 입력 페이로드를 파싱하고 사용자 메시지 추출
- 에이전트 초기화 및 세션 추적 관리
- 적절한 컨텍스트로 에이전트 호출 처리
- 런타임 환경에 형식화된 응답 반환

에이전트 파일을 생성해 보겠습니다:

In [ ]:
%%writefile runtime_memory_agent.py
import os
import json
import logging
from typing import Dict, Any
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
# 상세 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# 에이전트 코어 앱 초기화
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
REGION = os.getenv('AWS_REGION')

# 전역 에이전트 인스턴스 - 첫 번째 요청 시 초기화됨
agent = None
session_manager = None
 
   
    
def initialize_agent(actor_id, session_id):
    """메모리 세션 매니저로 에이전트 초기화"""
    global agent, session_manager

    logger.info(f"에이전트 초기화 중: actor_id={actor_id}, session_id={session_id}")

    # 모델 생성
    logger.info(f"모델 생성 중, ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)

    # AgentCoreMemoryConfig를 사용하여 메모리 구성 (권장)
    logger.info(f"메모리 설정 생성 중, 리전: {REGION}")
    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id=session_id,
        actor_id=actor_id
    )

    # 세션 매니저 생성 - 모든 메모리 작업을 자동으로 처리합니다!
    session_manager = AgentCoreMemorySessionManager(config, region_name=REGION)

    # session_manager로 에이전트 생성 (커스텀 훅을 대체)
    logger.info("AgentCoreMemorySessionManager로 에이전트 생성 중")
    # 시스템 프롬프트: AgentCore 런타임에 배포된 메모리 지원 에이전트, 같은 세션 내 이전 상호작용을 기억
    agent = Agent(
        model=model,
        session_manager=session_manager,  # 내장 메모리 통합
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses."
    )
    logger.info("메모리 세션 매니저로 에이전트 초기화 완료")

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    메모리 지원 에이전트의 메인 진입점
    
    Args:
        payload: 사용자 데이터가 포함된 입력 페이로드
        context: 세션 정보가 포함된 런타임 컨텍스트 객체
    """
    global agent, session_manager
    
    # 페이로드와 컨텍스트 정보 모두 로그 기록
    logger.info(f"수신된 페이로드: {payload}")
    logger.info(f"컨텍스트 session_id: {context.session_id}")
    
    # 필수 값 추출 및 검증
    user_input = payload.get("prompt")
    actor_id = payload.get("actor_id", "default_user")  # 데모용 기본값 제공
    session_id = context.session_id  # 컨텍스트에서 session_id 가져오기
    
    # 필수 필드 검증
    if user_input is None:
        error_msg = "오류: 페이로드에 'prompt' 필드가 없습니다"
        logger.error(error_msg)
        return error_msg
    
    # 첫 번째 요청 시 에이전트 초기화
    if agent is None:
        logger.info("첫 번째 요청 - 에이전트 초기화 중")
        initialize_agent(actor_id, session_id)
    else:
        
        # 세션 또는 액터가 변경되었는지 확인 - 재초기화 필요
        current_session = getattr(session_manager, '_session_id', None) if session_manager else None
        if current_session != session_id:
            logger.info(f"세션이 {current_session}에서 {session_id}로 변경됨 - 에이전트 재초기화 중")
            initialize_agent(actor_id, session_id)
    
    # 사용자 입력으로 에이전트 호출
    logger.info(f"에이전트 호출 중, 입력: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"에이전트 응답: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("AgentCore 애플리케이션 시작 중")
    app.run()

## 3. AgentCore 런타임에 배포

이 섹션에서는 에이전트를 Amazon Bedrock AgentCore 런타임(확장성과 간소화된 운영을 제공하는 관리형 에이전트 런타임 환경)에 배포합니다. AgentCore 런타임은 인프라의 복잡성을 처리하여, 배포 문제 대신 에이전트의 로직에 집중할 수 있게 합니다.

수동 서버 설정 및 관리가 필요한 기존 배포 방식과 달리, AgentCore 런타임은 코드를 자동으로 컨테이너로 패키징하고, AWS 인프라에 배포하며, 호출을 위한 안전한 HTTPS 엔드포인트를 제공합니다. 이 접근 방식은 에이전트가 수요에 따라 확장하고 프로덕션 환경에서 안정적으로 운영될 수 있도록 보장합니다.

### 알아야 할 사항

- **AgentCore 런타임**은 에이전트를 Docker 컨테이너로 패키징하여 관리형 AWS 인프라에 배포합니다
- **환경 변수**가 에이전트를 구성합니다:
  - `MEMORY_ID`: 앞서 생성한 메모리 리소스
  - `MODEL_ID`: Claude Haiku 4.5 모델 ID
  - `AWS_REGION`: 배포를 위한 AWS 리전

> **팁**: AgentCore 스타터 툴킷이 IAM 역할, ECR 저장소, 컨테이너 빌드를 포함한 모든 복잡한 배포 단계를 처리합니다.

### 배포 구성

배포 설정을 구성하겠습니다:

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_memory_agent.py", 
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    memory_mode="STM_ONLY",
)
response


### 메모리 리소스 등록

1단계에서는 AgentCore 메모리의 기본 구성 요소를 이해하기 위해 메모리 리소스를 수동으로 생성했습니다. 자체 메모리 리소스를 관리하고 있으므로, 배포 시 스타터 툴킷의 로컬 구성을 업데이트하여 이를 사용하도록 해야 합니다. 이렇게 하면 툴킷이 새로운 메모리 리소스를 프로비저닝하는 것을 방지합니다.

다음 셀은 `configure()`에서 생성된 `.bedrock_agentcore.yaml` 구성 파일에 메모리 리소스 ID를 기록합니다.

> **참고**: AgentCore 스타터 툴킷을 사용할 때는 메모리 리소스를 수동으로 관리할 필요가 없습니다. AgentCore 스타터 툴킷은 `configure()` 시 `memory_mode`를 설정하면 메모리 프로비저닝, 구성 및 생명주기 관리를 자동으로 처리합니다. 이 튜토리얼에서는 각 구성 요소가 어떻게 작동하는지 전체적으로 볼 수 있도록 리소스를 명시적으로 생성합니다.

In [ ]:
import yaml

config_path = ".bedrock_agentcore.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# 기존 memory_id를 에이전트 설정에 주입
config['agents'][agent_name]['memory']['memory_id'] = memory_id

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"memory_id: {memory_id}를 {agent_name} 설정에 주입했습니다")

### 에이전트 시작

이제 에이전트를 AgentCore 런타임에 시작합니다. 이 단계에서는 구성된 에이전트를 가져와 AgentCore의 관리형 인프라에 배포합니다. 이 과정에서 에이전트에 필요한 필수 환경 변수(앞서 생성한 메모리 ID와 사용할 모델 ID)도 전달합니다. 배포가 완료되면 사용자 메시지로 호출할 수 있는 안전한 엔드포인트를 통해 에이전트에 접근할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": memory_id,
        "MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    }
)

### 배포 상태 확인

에이전트의 배포 상태를 확인하겠습니다:

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"현재 상태: {status}")

if status == 'READY':
    print("에이전트가 성공적으로 배포되었습니다!")
else:
    print(f"배포가 다음 상태로 종료됨: {status}")

## 에이전트 테스트

에이전트가 배포되었으니, 메시지를 보내서 테스트해 보겠습니다.

**액터 ID와 세션 관리에 대한 중요 사항**

- 액터 ID: 프로덕션 애플리케이션에서 actor_id는 일반적으로 사용자 로그인 시 인증 시스템에서 가져옵니다. 이 식별자는 에이전트가 다른 사용자에 대한 별도의 대화 이력을 유지하는 데 도움이 됩니다. 이 예제에서는 하드코딩된 값(test_user_123)을 사용하지만, 실제 시나리오에서는 인증된 사용자의 고유 식별자를 전달합니다.

- 세션 관리: AgentCore 런타임은 세션 ID가 제공되지 않으면 자동으로 생성하지만, 애플리케이션에서 세션 ID를 명시적으로 관리하는 것이 권장됩니다. 이를 통해 다음을 더 잘 제어할 수 있습니다:
  - 세션 타임아웃 후 대화 계속
  - 적절한 시점에 새 세션 생성 (예: 사용자가 새 대화를 시작하는 경우)
  - 동일한 사용자와의 여러 병렬 대화 처리
  - 애플리케이션의 필요에 따른 세션 만료 정책 구현

In [ ]:
# 테스트 세션 ID 생성
test_session_id = "agent-runtime-memory-session-123456789" # 최소 길이 33자

# 첫 번째 메시지 전송 (인사 및 이름 알려주기 + 기능 질문)
invoke_response = agentcore_runtime.invoke(
    {
        "prompt": "Hello! My name is John. What can you do?",
        "actor_id": "test_user_123"
    }, 
    session_id=test_session_id
)

invoke_response

### 에이전트 응답 표시

응답을 더 읽기 쉬운 형식으로 표시하겠습니다:

In [ ]:
from IPython.display import Markdown, display
import json

response_text = invoke_response['response'][0]
display(Markdown(response_text))

### 영속성 테스트

이제 동일한 세션에서 후속 메시지를 보내 에이전트가 이전 상호작용을 기억하는지 테스트해 보겠습니다:

In [ ]:
# 동일한 세션 ID를 사용하여 후속 메시지 전송 (이름을 기억하는지 확인)
follow_up_response = agentcore_runtime.invoke(
    {
        "prompt": "What is my name?",
        "actor_id": "test_user_123"
    }, 
    session_id=test_session_id
)

# 응답 표시
follow_up_text = follow_up_response['response'][0]
display(Markdown(follow_up_text))

### 메모리 내용 확인

메모리에 저장된 내용을 확인하여 메시지가 제대로 저장되었는지 확인하겠습니다:

In [ ]:
# 세션 작업을 위해 MemorySessionManager 사용
manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
session = manager.create_memory_session(
    actor_id="test_user_123",
    session_id=test_session_id
)

# 세션 범위 메서드를 사용하여 대화 이력 가져오기
stored_turns = session.get_last_k_turns(k=10)

print(f"메모리에서 {len(stored_turns)}개의 대화 턴을 찾았습니다 (시간순으로 표시):")
for idx, turn in enumerate(stored_turns):
    print(f"\n턴 {idx+1}:")
    for message in turn:
        role = message['role']
        text = message['content']['text']
        print(f"- {role}: {text[:100]}...")

## 핵심 개념

1. **메모리 통합**: Amazon Bedrock 메모리를 사용하여 대화 이력을 저장하는 방법
2. **세션 관리**: 세션 ID를 사용하여 대화 컨텍스트를 유지하는 방법
3. **AgentCore 배포**: 프로덕션 런타임 환경에 에이전트를 배포하는 방법
4. **AgentCoreMemorySessionManager**: 자동 메모리 처리를 위한 내장 Strands 통합 사용 방법

이러한 개념은 영속적인 메모리를 갖춘 더 복잡한 에이전트를 구축하기 위한 기반을 제공합니다.

## 정리 (선택 사항)

이 튜토리얼에서 생성한 리소스가 더 이상 필요하지 않으면 정리할 수 있습니다:

In [ ]:
# 리소스 식별자 가져오기
if 'launch_result' in locals():
    print(f"에이전트 ID: {launch_result.agent_id}")
    print(f"ECR 저장소: {launch_result.ecr_uri.split('/')[1]}")
else:
    print("시작 결과를 사용할 수 없습니다")

In [ ]:
# 리소스를 삭제하려면 이 셀을 실행하세요

# AgentCore 런타임 삭제
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=REGION
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)
print(f"AgentCore 런타임 삭제 완료: {launch_result.agent_id}")

# ECR 저장소 삭제
ecr_client = boto3.client(
    'ecr',
    region_name=REGION
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1].split(':')[0],
    force=True
)
print(f"ECR 저장소 삭제 완료: {launch_result.ecr_uri.split('/')[1].split(':')[0]}")

# 메모리 리소스 삭제
memory_client = MemoryClient(region_name=REGION)
memory_client.delete_memory_and_wait(memory_id=memory_id)
print(f"메모리 리소스 삭제 완료: {memory_id}")

## 축하합니다!

Amazon Bedrock AgentCore 런타임과 AgentCore 메모리를 사용한 첫 번째 메모리 지원 에이전트를 성공적으로 구축하고 배포했습니다!

### 다음 단계

기본 사항을 이해했으니, 다음을 시도할 수 있습니다:

1. **도구 추가**: 계산기, 데이터베이스 커넥터 또는 API 호출과 같은 도구로 에이전트 향상
2. **메모리 개선**: 장기 메모리를 활용한 더 정교한 메모리 전략 구현
3. **UI 구축**: 에이전트를 위한 웹 또는 모바일 인터페이스 생성